Start from the dataset kickstarter-14-04, take out the unnecessary columsn that I don't know why are there:

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import nltk
from nltk.corpus import stopwords
import string 
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
import re 
from collections import Counter
import ast
import glob

In [2]:
campaigns = pd.read_csv('raw_kickstarter.csv', index_col=0)
campaigns

,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency
0,https://www.kickstarter.com/projects/thetruthb...,NaN,The PROBLEM: So much entertainment today pushe...,71123.0,71123.0,61607,48000.0,USD
1,https://www.kickstarter.com/projects/99625582/...,NaN,Millions of American college students have stu...,65318.0,65318.0,56579,61500.0,USD
2,https://www.kickstarter.com/projects/distortre...,Cartoon Network Alphabet Pins,Full A-Z Set I'm launching this set to show my...,462.0,462.0,400,8000.0,USD
3,https://www.kickstarter.com/projects/jordym/th...,The Balloon - a short film,"On a sleepy summer afternoon, we stared into t...",5137.0,5137.0,4449,15000.0,USD
4,https://www.kickstarter.com/projects/trans-mov...,NaN,48 hours of pledge matching! Amazing news! Two...,50640.0,50640.0,43865,50000.0,USD
...,...,...,...,...,...,...,...,...
8390,https://www.kickstarter.com/projects/zerwcolla...,ZerwCollab: Your AI-Powered App Builder in Sec...,🚀 What If You Could Launch a Website or App… U...,2128.0,2128.0,1823,35900.0,USD
8391,https://www.kickstarter.com/projects/jacobzirk...,VFX Oasis: Free VFX Assets and Footage,What is VFX Oasis? VFX Oasis is an industry fo...,580.0,580.0,496,6000.0,USD
8392,https://www.kickstarter.com/projects/153189407...,Sharp X - Edge Gadget,Kickstarter Project Story – USA-Market Sharpen...,857.0,857.0,734,7800.0,USD
8393,https://www.kickstarter.com/projects/diyengine...,Steady Shot Bot: Hyperlapse + Steadycam + Auto...,"Why As a hobbyist, I recently found myself wan...",6645.0,6645.0,5693,310000.0,USD


### as a very first step we add the categories to each project

In [3]:

folder_path = "Kickstarter_2026-03-12T03_20_26_556Z"

files = glob.glob(f"{folder_path}/*.csv")

print("Number of files found:", len(files))

dfs = []

for file in files:
    print("Loading:", file)
    df_temp = pd.read_csv(file)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)



url_to_category = {}
for _, row in df.iterrows():
    parsed = ast.literal_eval(row['category'])
    parent_name = parsed.get('parent_name') or parsed.get('name')
    url_parsed = ast.literal_eval(row['urls'])
    project_url = url_parsed['web']['project']
    url_to_category[project_url] = parent_name


campaigns['category'] = campaigns['url'].map(url_to_category)

print(f"Matched: {campaigns['category'].notna().sum()} / {len(campaigns)}")
print(campaigns['category'].value_counts())

Number of files found: 85
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter001.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter002.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter003.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter004.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter005.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter006.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter007.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter008.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter009.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter010.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter011.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter012.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter013.csv
Loading: Kickstarter_2026-03-12T03_20_26_556Z\Kickstarter014.csv
Lo

take out the unnecessary columns, add the reached one and classify as success or failure (maybe it's already in the initial dataset but honestly I don't remember)

In [4]:
campaigns['reached'] = (campaigns['pledged'] / campaigns['goal']) * 100

median_reached = campaigns['reached'].median()
campaigns['status'] = campaigns['reached'].apply(lambda x: 1 if x >= median_reached else 0)

In [5]:
campaigns

,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency,category,reached,status
0,https://www.kickstarter.com/projects/thetruthb...,NaN,The PROBLEM: So much entertainment today pushe...,71123.0,71123.0,61607,48000.0,USD,Film & Video,148.172917,1
1,https://www.kickstarter.com/projects/99625582/...,NaN,Millions of American college students have stu...,65318.0,65318.0,56579,61500.0,USD,Film & Video,106.208130,1
2,https://www.kickstarter.com/projects/distortre...,Cartoon Network Alphabet Pins,Full A-Z Set I'm launching this set to show my...,462.0,462.0,400,8000.0,USD,Film & Video,5.775000,0
3,https://www.kickstarter.com/projects/jordym/th...,The Balloon - a short film,"On a sleepy summer afternoon, we stared into t...",5137.0,5137.0,4449,15000.0,USD,Film & Video,34.246667,0
4,https://www.kickstarter.com/projects/trans-mov...,NaN,48 hours of pledge matching! Amazing news! Two...,50640.0,50640.0,43865,50000.0,USD,Film & Video,101.280000,0
...,...,...,...,...,...,...,...,...,...,...,...
8390,https://www.kickstarter.com/projects/zerwcolla...,ZerwCollab: Your AI-Powered App Builder in Sec...,🚀 What If You Could Launch a Website or App… U...,2128.0,2128.0,1823,35900.0,USD,Technology,5.927577,0
8391,https://www.kickstarter.com/projects/jacobzirk...,VFX Oasis: Free VFX Assets and Footage,What is VFX Oasis? VFX Oasis is an industry fo...,580.0,580.0,496,6000.0,USD,Technology,9.666667,0
8392,https://www.kickstarter.com/projects/153189407...,Sharp X - Edge Gadget,Kickstarter Project Story – USA-Market Sharpen...,857.0,857.0,734,7800.0,USD,Technology,10.987179,0
8393,https://www.kickstarter.com/projects/diyengine...,Steady Shot Bot: Hyperlapse + Steadycam + Auto...,"Why As a hobbyist, I recently found myself wan...",6645.0,6645.0,5693,310000.0,USD,Technology,2.143548,0


Preprocessing: usual preprocessing stuff like lowercasing, taking out links, only keepin alpha numeric characters, tokenizing the words, and taking out the english stopwords, also took out the words that are less than 2 characters

In [6]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text_p = "".join([char for char in text if char.isalnum() or char.isspace()])
    
    words = word_tokenize(text_p) 
    
    
    stop_words = stopwords.words('english')
    words = [w for w in words if w not in stop_words and len(w) > 2 and w.isalpha()]
    filtered_words = [word for word in words if word not in stop_words] 
    
    return filtered_words  

campaigns['description_processed'] = campaigns['description'].apply(preprocess)

How do we decide which stopwords to include beside the 'normal' stopwords? We also want to include words that are very generic, not informative at all and coudl potentially appear in any kind of campaign, regardless of the topics in it (some examples could be the words 'kickstarter', campaign, etc etc)
Initially I thought TF-IDF would make sense but it doesn't actually, because TF-IDF both depends on how frequent a word is in each document and how frequent are documents that contain that words in the whole set of documents. (also you cannot average the TF-IDF of a word and pick the ones with lowest value because a rare word, which we wanna keep, could end up having a low TF-IDF value if it appears in few documents but many time in those few documents)
So potentially we can just use document frequency, meaning if the word appears in > alpha% of documents, we count it as a stopword?

Also quick sidenote: Other methods like BytePair encoding, WordPiece or SentencePiece are not really useful here because they look at subwords, and for example if we get 'kickstarter' and split it into kick and starter, then we might consider to keep kick because it makes sense with sport but we might not have any document where the word is present for real

This below just basically gives a counter of the document frequency of each token in our whole corpus of all the descriptions 

We apply lemmatization (I think probably better than stemming since it returns the lemma version of the word, instead of just cutting the ending of the word). We apply it now, before taking out the domain specific stopwords, because otherwise we could end up having to account for different versions of the same stopword (example: project vs projects)

In [7]:
lemmatizer = WordNetLemmatizer()

campaigns['lemmatized'] = campaigns['description_processed'].apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words])


In [8]:
initial_vocab = set(token for doc in campaigns['lemmatized'] for token in doc)
print(f"lenght of the initial vocabulary: {len(initial_vocab)}")

lenght of the initial vocabulary: 111122


In [9]:
docs = campaigns['lemmatized']
N = len(docs)

df_counter = Counter()

for doc in docs:
    df_counter.update(set(doc))

df_table = pd.DataFrame({
    'word': list(df_counter.keys()),
    'doc_freq': list(df_counter.values())
})

df_table['doc_freq_ratio'] = df_table['doc_freq'] / N
df_table = df_table.sort_values('doc_freq_ratio', ascending=False)



In [10]:
print(f"total number of tokens is: {len(df_table)}")
df_table


total number of tokens is: 111122


,word,doc_freq,doc_freq_ratio
115,one,5194,0.706282
13,make,5190,0.705738
388,time,5127,0.697172
16,help,4925,0.669704
237,project,4743,0.644955
...,...,...,...
111111,eyework,1,0.000136
111110,mussing,1,0.000136
111109,dbu,1,0.000136
111108,noncontinuous,1,0.000136


Now the only method and the one that made most sense to me to take out too common words is basically just defining a threshold and see which words are too rare (like if it appears only 2-3 times in the whole dataset) or too much (in this case too much can be like appearing in more than 40-60% of the descriptions), you can try different values of the two initial thresholds (the one I saw looks the best would be 0.0005 for the low and 0.55 for the high)

In [11]:
min_doc_count = 5
min_ratio = min_doc_count / N
top_n_words = 100


top_words_preview = (
    df_table
    .sort_values('doc_freq_ratio', ascending=False)
    .head(top_n_words)
)

print(df_table[df_table['doc_freq_ratio'] <= min_ratio])
top_words_preview




                 word  doc_freq  doc_freq_ratio
27874           única         5        0.000680
31613         avenged         5        0.000680
21863            homo         5        0.000680
42164          regent         5        0.000680
42160         detract         5        0.000680
...               ...       ...             ...
111111        eyework         1        0.000136
111110        mussing         1        0.000136
111109            dbu         1        0.000136
111108  noncontinuous         1        0.000136
111107       adatsmux         1        0.000136

[88937 rows x 3 columns]


,word,doc_freq,doc_freq_ratio
115,one,5194,0.706282
13,make,5190,0.705738
388,time,5127,0.697172
16,help,4925,0.669704
237,project,4743,0.644955
...,...,...,...
822,please,2158,0.293446
2895,hope,2151,0.292494
53,film,2131,0.289774
318,used,2122,0.288550


In [12]:
vocab = set(
    df_table['word']
)
print(f"length of initial vocabulary is {len(vocab)}")

rare_words = set(df_table['word'][df_table['doc_freq_ratio'] <= min_ratio] )


common_words = set(top_words_preview["word"])
vocab = vocab - common_words - rare_words
print(f"there are {len(common_words)} common words")
print(f"there are {len(rare_words)} rare words")
print(f"Removed {len(rare_words) + len(common_words)} words from vocab")
print(f"the vocabulary size now after having took out the general stopwords (not category specific) is {len(vocab)} tokens")
vocab

length of initial vocabulary is 111122
there are 100 common words
there are 88937 rare words
Removed 89037 words from vocab
the vocabulary size now after having took out the general stopwords (not category specific) is 22085 tokens


{'succulent',
 'submit',
 'brochure',
 'yielded',
 'suburb',
 'florentine',
 'strength',
 'outside',
 'fad',
 'type',
 'anguish',
 'simplifies',
 'crackling',
 'vic',
 'sweetwater',
 'reducing',
 'doable',
 'sustaining',
 'explorando',
 'bohemian',
 'shelley',
 'astral',
 'perilous',
 'complex',
 'meticulously',
 'stitcher',
 'thermoplastic',
 'better',
 'synth',
 'tidbit',
 'weaken',
 'necessity',
 'screwdriver',
 'bookworm',
 'hardware',
 'unexpected',
 'coat',
 'unconfirmed',
 'reinvention',
 'loss',
 'journalist',
 'projectile',
 'chuckle',
 'congested',
 'optimize',
 'advert',
 'onslaught',
 'neo',
 'shoutout',
 'havoc',
 'outperforms',
 'stimulates',
 'architecture',
 'culminated',
 'whirlwind',
 'ascended',
 'photography',
 'browne',
 'perfume',
 'horrific',
 'punched',
 'entity',
 'lived',
 'garden',
 'mecha',
 'misstep',
 'davenport',
 'whats',
 'edits',
 'coexecutive',
 'yellowing',
 'alley',
 'lavender',
 'forgive',
 'reawaken',
 'rebellion',
 'obscure',
 'helium',
 'mattel'

### Now we work on the data specific category by category 

In [13]:
# to actually take the overall common words out CAREFUL TO RUN 
campaigns['description_processed'] = campaigns['lemmatized'].apply(
    lambda doc: [w for w in doc if w in vocab]
)
campaigns = campaigns.drop(columns=['lemmatized'])

In [14]:
df = campaigns

In [15]:
df

,url,title,description,pledged,usd_pledged,converted_pledged_amount,goal,currency,category,reached,status,description_processed
0,https://www.kickstarter.com/projects/thetruthb...,NaN,The PROBLEM: So much entertainment today pushe...,71123.0,71123.0,61607,48000.0,USD,Film & Video,148.172917,1,"[problem, entertainment, today, push, unholy, ..."
1,https://www.kickstarter.com/projects/99625582/...,NaN,Millions of American college students have stu...,65318.0,65318.0,56579,61500.0,USD,Film & Video,106.208130,1,"[million, american, college, student, studied,..."
2,https://www.kickstarter.com/projects/distortre...,Cartoon Network Alphabet Pins,Full A-Z Set I'm launching this set to show my...,462.0,462.0,400,8000.0,USD,Film & Video,5.775000,0,"[launching, early, cartoon, network, started, ..."
3,https://www.kickstarter.com/projects/jordym/th...,The Balloon - a short film,"On a sleepy summer afternoon, we stared into t...",5137.0,5137.0,4449,15000.0,USD,Film & Video,34.246667,0,"[sleepy, summer, afternoon, stared, void, floa..."
4,https://www.kickstarter.com/projects/trans-mov...,NaN,48 hours of pledge matching! Amazing news! Two...,50640.0,50640.0,43865,50000.0,USD,Film & Video,101.280000,0,"[hour, pledge, matching, amazing, news, genero..."
...,...,...,...,...,...,...,...,...,...,...,...,...
8390,https://www.kickstarter.com/projects/zerwcolla...,ZerwCollab: Your AI-Powered App Builder in Sec...,🚀 What If You Could Launch a Website or App… U...,2128.0,2128.0,1823,35900.0,USD,Technology,5.927577,0,"[launch, website, app, using, voice, imagine, ..."
8391,https://www.kickstarter.com/projects/jacobzirk...,VFX Oasis: Free VFX Assets and Footage,What is VFX Oasis? VFX Oasis is an industry fo...,580.0,580.0,496,6000.0,USD,Technology,9.666667,0,"[vfx, oasis, vfx, oasis, industry, focused, vi..."
8392,https://www.kickstarter.com/projects/153189407...,Sharp X - Edge Gadget,Kickstarter Project Story – USA-Market Sharpen...,857.0,857.0,734,7800.0,USD,Technology,10.987179,0,"[real, today, modern, especially, continues, l..."
8393,https://www.kickstarter.com/projects/diyengine...,Steady Shot Bot: Hyperlapse + Steadycam + Auto...,"Why As a hobbyist, I recently found myself wan...",6645.0,6645.0,5693,310000.0,USD,Technology,2.143548,0,"[hobbyist, recently, found, wanting, photograp..."


#### 1. Identification of words appearing in more than 40% of documents

In [16]:

selected_categories = ['Technology', 'Games', 'Music', 'Publishing', 'Film & Video']
top_n_words = 50
category_high_freq_words = {}

for category_name in selected_categories:
    category_df = df[df["category"] == category_name].copy()
    docs = category_df["description_processed"]
    
    category_counter = Counter()
    for doc in docs:
        category_counter.update(set(doc))

    category_table = pd.DataFrame({
        "word": list(category_counter.keys()),
        "doc_freq": list(category_counter.values())
    })

    category_table["doc_freq_ratio"] = category_table["doc_freq"] / len(docs)
    high_freq_words = category_table.sort_values("doc_freq_ratio", ascending=False).head(top_n_words)

    category_high_freq_words[category_name] = high_freq_words
    print(f"\n{category_name}")
    display(high_freq_words)

category_high_freq_words


Technology


,word,doc_freq,doc_freq_ratio
196,product,854,0.573539
291,designed,802,0.538617
66,using,746,0.501007
71,technology,701,0.470786
317,system,672,0.451310
169,user,667,0.447952
177,control,618,0.415044
105,device,611,0.410343
285,easy,601,0.403627
117,power,597,0.400940



Games


,word,doc_freq,doc_freq_ratio
81,game,1199,0.843179
1101,player,864,0.607595
285,backer,667,0.469058
379,character,649,0.456399
438,stretch,645,0.453586
629,end,630,0.443038
1190,fun,624,0.438819
658,add,617,0.433896
648,shipping,613,0.431083
458,pledge,608,0.427567



Music


...


Publishing


,word,doc_freq,doc_freq_ratio
126,book,938,0.772652
634,page,583,0.480231
75,cover,557,0.458814
723,print,520,0.428336
86,author,506,0.416804
1054,copy,472,0.388797
849,read,456,0.375618
318,word,442,0.364086
43,writing,441,0.363262
18,reader,423,0.348435



Film & Video


,word,doc_freq,doc_freq_ratio
130,director,1024,0.509199
179,short,995,0.494779
85,crew,959,0.476877
419,producer,934,0.464446
453,festival,896,0.445549
988,character,842,0.418697
219,movie,825,0.410244
224,cast,822,0.408752
787,money,756,0.375932
345,series,750,0.372949


{'Technology':              word  doc_freq  doc_freq_ratio
 196       product       854        0.573539
 291      designed       802        0.538617
 66          using       746        0.501007
 71     technology       701        0.470786
 317        system       672        0.451310
 169          user       667        0.447952
 177       control       618        0.415044
 105        device       611        0.410343
 285          easy       601        0.403627
 117         power       597        0.400940
 11    development       580        0.389523
 264         build       571        0.383479
 605        better       570        0.382807
 310          tool       548        0.368032
 276        allows       539        0.361988
 296           app       533        0.357958
 392        simple       529        0.355272
 150     prototype       524        0.351914
 931         start       521        0.349899
 100         built       514        0.345198
 87          offer       509        0.341

In [17]:
category_stopwords_all = set()

for category_name, high_freq_words in category_high_freq_words.items():
    category_words = high_freq_words["word"].tolist()
    category_stopwords_all.update(category_words)
    print(f"\n{category_name} ({len(category_words)} words)")
    print(category_words)

print(f"\nTotal unique category stopwords: {len(category_stopwords_all)}")
category_stopwords_all


Technology (50 words)
['product', 'designed', 'using', 'technology', 'system', 'user', 'control', 'device', 'easy', 'power', 'development', 'build', 'better', 'tool', 'allows', 'app', 'simple', 'prototype', 'start', 'built', 'offer', 'quality', 'process', 'ready', 'provide', 'high', 'change', 'solution', 'idea', 'company', 'youre', 'backer', 'content', 'free', 'platform', 'access', 'future', 'easily', 'based', 'perfect', 'created', 'possible', 'software', 'already', 'building', 'market', 'browser', 'hand', 'problem', 'thats']

Games (50 words)
['game', 'player', 'backer', 'character', 'stretch', 'end', 'fun', 'add', 'shipping', 'pledge', 'designed', 'system', 'additional', 'youre', 'using', 'card', 'based', 'final', 'build', 'complete', 'start', 'playing', 'special', 'content', 'order', 'gameplay', 'always', 'idea', 'change', 'choose', 'book', 'free', 'already', 'creating', 'product', 'version', 'tier', 'adventure', 'development', 'everything', 'include', 'everyone', 'includes', 'crea

{'access',
 'actor',
 'add',
 'additional',
 'adventure',
 'album',
 'allows',
 'along',
 'already',
 'always',
 'app',
 'audience',
 'author',
 'award',
 'backer',
 'band',
 'based',
 'beautiful',
 'believe',
 'better',
 'big',
 'book',
 'box',
 'browser',
 'budget',
 'build',
 'building',
 'built',
 'camera',
 'cant',
 'card',
 'cast',
 'change',
 'character',
 'choose',
 'city',
 'company',
 'complete',
 'content',
 'control',
 'copy',
 'cover',
 'created',
 'creating',
 'creative',
 'crew',
 'currently',
 'designed',
 'development',
 'device',
 'digital',
 'director',
 'dream',
 'easily',
 'easy',
 'edition',
 'end',
 'engineer',
 'equipment',
 'ever',
 'everyone',
 'everything',
 'excited',
 'fan',
 'festival',
 'filmmaker',
 'final',
 'free',
 'fun',
 'future',
 'game',
 'gameplay',
 'going',
 'good',
 'guitar',
 'hand',
 'hear',
 'heart',
 'high',
 'home',
 'idea',
 'include',
 'includes',
 'ive',
 'join',
 'journey',
 'last',
 'little',
 'live',
 'location',
 'long',
 'lot',
 '

In [18]:
vocab = vocab - category_stopwords_all
print(f"final vocabulary length: {len(vocab)}")
df["description_processed"] = df["description_processed"].apply(
    lambda doc: [word for word in doc if word in vocab]
)

final vocabulary length: 21911


In [19]:
# only at the end 
df.to_csv('Kickstarter_processed.csv')